# Notebook 11 — Pathway-Based External Validation (RNA-Seq → Microarray)

**Problem:** gene-level models trained on TCGA-PRAD do not transfer to GSE70769 (AUC 0.43–0.53). Single-gene measurements are platform- and cohort-specific.

**Solution:** aggregate genes into pre-specified **pathway scores** (mean expression of curated gene sets). Pathway-level biology is far more stable across platforms — this is exactly how the commercial assays (Prolaris/CCP, Decipher) achieve cross-platform validity.

1. 15 pre-specified pathway scores from literature gene sets (no data-driven selection)
2. PRIMARY analysis: all 15 pathways, regularized logistic, C tuned on internal CV only
3. SECONDARY analysis: the 5 literature-established BCR-signature pathways (Prolaris CCP, Decipher, Cell_Cycle, DNA_Repair, Proliferation)
4. Bootstrap 95% CIs + p-value vs AUC=0.5
5. Triangulation on GSE54460 (2nd external cohort, RNA-Seq FFPE)

In [5]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

warnings.filterwarnings("ignore")
CORE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED, INTERIM = CORE / "data" / "processed", CORE / "data" / "interim"
TABLES = CORE / "outputs" / "tables"

X_train = pd.read_csv(PROCESSED / "X_train_preprocessed.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").iloc[:, 0]
X_test  = pd.read_csv(PROCESSED / "X_test_preprocessed.csv")
y_test  = pd.read_csv(PROCESSED / "y_test.csv").iloc[:, 0]
X_ext   = pd.read_csv(PROCESSED / "X_GSE70769.csv", index_col=0)
y_ext   = pd.read_csv(PROCESSED / "y_GSE70769.csv", index_col=0).iloc[:, 0]
X_ext   = X_ext.loc[y_ext.index]

# GSE54460 (rebuild from raw if interim missing)
if not (INTERIM / "X_GSE54460.csv").exists():
    raw = pd.read_csv(CORE / "data" / "external" / "GSE54460_FPKM.txt.gz", sep="\t", low_memory=False)
    g = raw.loc[raw[raw.iloc[:, 0] == "Entrez"].index[0] + 1:].copy()
    g.columns = raw.columns
    g = g.set_index(g.columns[1]).iloc[:, 1:]
    g = g.loc[g.index.notna()]
    g = g[~g.index.duplicated(keep="first")].apply(pd.to_numeric, errors="coerce").fillna(0)
    bcr = raw[raw.iloc[:, 0] == "BCR"].iloc[0, 2:].values.astype(int)
    m = g.T; m.index = raw.columns[2:]; m["bcr"] = bcr
    m["patient"] = [s.rsplit(".", 1)[0] if "." in s else s for s in m.index]
    yb = m.groupby("patient")["bcr"].first()
    Xp = np.log2(m.groupby("patient").mean().drop(columns="bcr") + 1)
    Xp.to_csv(INTERIM / "X_GSE54460.csv"); yb.to_csv(INTERIM / "y_GSE54460.csv")
X_54460 = pd.read_csv(INTERIM / "X_GSE54460.csv", index_col=0)
y_54460 = pd.read_csv(INTERIM / "y_GSE54460.csv", index_col=0).iloc[:, 0]

print(f"Train {X_train.shape} (+{int(y_train.sum())}), Test {X_test.shape} (+{int(y_test.sum())})")
print(f"GSE70769 {X_ext.shape} (+{int(y_ext.sum())}), GSE54460 {X_54460.shape} (+{int(y_54460.sum())})")

Train (343, 19019) (+46), Test (86, 19019) (+12)
GSE70769 (94, 29720) (+45), GSE54460 (46, 23281) (+34)


In [6]:
# --- 15 pre-specified pathway gene sets (literature) ---
DECIPHER = ['CEACAM1','FLNA','HES6','KPNA2','LCP1','PLA2G7','PTGER4','RAB25','SAA1','SORD','STOM','TPX2','TUBE1','PDSS2','SELENBP1','SRD5A2','TP53BP1']
PROLARIS = ['BIRC5','CDC20','CDKN1A','CENPF','DUSP6','EZH2','FOXM1','GTSE1','KLK2','KIF11','KIF14','KIF20A','MCM2','MCM5','MCM7','MKI67','NDC80','PCNA','PLK1','PTTG1','RRM2','SPP1','TOP2A','AURKA','AURKB','BUB1','BUB1B','CCNB1','CCNB2','CDCA3','CDKN3','CENPE','CENPN','DLGAP5','EXO1','GAS6','HMMR','KIF2C','KIF4A','MELK','NCAPD2','NUF2','PBK','RACGAP1','RFC4','TK1','UBE2C','ZWINT']
AR_SIG = ['AR','KLK3','KLK2','TMPRSS2','FKBP5','STEAP2','ACPP','CAMKK2']
EMT = ['CDH1','VIM','CDH2','SNAI1','SNAI2','ZEB1','FN1','CD44','ITGA6']
PROLIF = ['MKI67','TOP2A','PCNA','MCM2','MCM5','MCM7','AURKA','BIRC5','CCNB1']
DNA_REPAIR = ['BRCA1','BRCA2','RAD51','ATM','CHEK2','XRCC2','PARP1','PALB2','RAD54L','GEN1']
PI3K_AKT = ['PTEN','PIK3CA','AKT1','MTOR','RPS6KB1','EIF4EBP1','PDK1','TSC1','TSC2']
ANDROGEN_R = ['AR','KLK3','KLK2','TMPRSS2','SRD5A2','HSD3B1','CYP17A1']
CELL_CYCLE = ['CCND1','CCNE1','CDK2','CDK4','CDK6','RB1','E2F1','TP53','CDKN1A','CDKN2A','CDKN1B']
STROMA = ['ACTA2','COL1A1','COL3A1','FAP','PDGFRB','POSTN','THBS1','TGFBI','TAGLN','VIM']
IMMUNE = ['CD68','CD8A','CD4','FOXP3','PDCD1','CTLA4','LAG3','CD274','IFNG','GZMB']
HYPOXIA = ['HIF1A','VEGFA','CA9','EGLN1','EGLN3','SLC2A1','LOX','P4HA1','LDHA']
METABOLISM = ['SLC2A1','HK2','PKM','LDHA','ACLY','FASN','SCD','ACACA','HMGCS2','CPT1A']
WNT_BETA = ['CTNNB1','APC','AXIN2','LEF1','TCF7','MYC','CCND1','DVL2','FRAT1','WNT5A']
STRESS = ['HSP90AA1','HSP90AB1','HSPA5','HSPA8','HSPB1','HSPD1','HSPE1','DNAJA1','DNAJB1','STIP1']

ALL_PATHWAYS = {
    'Decipher': DECIPHER, 'Prolaris': PROLARIS, 'AR_Signaling': AR_SIG,
    'EMT': EMT, 'Proliferation': PROLIF, 'DNA_Repair': DNA_REPAIR,
    'PI3K_AKT': PI3K_AKT, 'Androgen_Response': ANDROGEN_R,
    'Cell_Cycle': CELL_CYCLE, 'Stroma': STROMA, 'Immune': IMMUNE,
    'Hypoxia': HYPOXIA, 'Metabolism': METABOLISM,
    'WNT_Beta_Catenin': WNT_BETA, 'Stress_Response': STRESS,
}
# Secondary pre-specified subset: the published BCR-prognostic signatures
BCR_LITERATURE = ['Prolaris', 'Decipher', 'Cell_Cycle', 'DNA_Repair', 'Proliferation']

def pscores(df, sets):
    s = pd.DataFrame(index=df.index)
    for name, genes in sets.items():
        p = [g for g in genes if g in df.columns]
        if len(p) >= 2:
            s[name] = df[p].mean(axis=1)
    return s

ps_t  = pscores(X_train, ALL_PATHWAYS)
ps_te = pscores(X_test, ALL_PATHWAYS)
ps_e  = pscores(X_ext, ALL_PATHWAYS)
ps_5  = pscores(X_54460, ALL_PATHWAYS)
print(f"Pathway scores built: {ps_t.shape[1]} pathways")
display(ps_t.describe().T[['mean', 'std']].round(2))

Pathway scores built: 15 pathways


,mean,std
Decipher,0.0,0.31
Prolaris,0.0,0.25
AR_Signaling,-0.0,0.39
EMT,-0.0,0.28
Proliferation,-0.0,0.30
DNA_Repair,-0.0,0.40
PI3K_AKT,-0.0,0.37
Androgen_Response,-0.0,0.39
Cell_Cycle,0.0,0.28
Stroma,-0.0,0.35


In [7]:
def bootstrap_ci(y, prob, n_boot=3000, seed=42):
    rng = np.random.RandomState(seed)
    yv = np.asarray(y)
    boots = []
    for _ in range(n_boot):
        idx = rng.choice(len(yv), size=len(yv), replace=True)
        if len(np.unique(yv[idx])) < 2:
            continue
        boots.append(roc_auc_score(yv[idx], prob[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min(np.mean(np.array(boots) <= 0.5), np.mean(np.array(boots) >= 0.5))
    return lo, hi, p

def eval_config(cols, label):
    # C tuned on INTERNAL CV only — external touched once
    best_C, best_cv = 0.1, -1
    for C in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]:
        lr = LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        cv = cross_val_score(lr, ps_t[cols], y_train, cv=5, scoring="roc_auc").mean()
        if cv > best_cv:
            best_cv, best_C = cv, C
    lr = LogisticRegression(C=best_C, max_iter=2000, class_weight="balanced")
    lr.fit(ps_t[cols], y_train)
    out = {"config": label, "C": best_C, "cv_auc": best_cv,
           "test_auc": roc_auc_score(y_test, lr.predict_proba(ps_te[cols])[:, 1])}
    for cohort, Xc, yc in [("GSE70769", ps_e, y_ext), ("GSE54460", ps_5, y_54460)]:
        p = lr.predict_proba(Xc[cols])[:, 1]
        auc = roc_auc_score(yc, p)
        lo, hi, pv = bootstrap_ci(yc, p)
        out[f"{cohort}_auc"], out[f"{cohort}_ci"] = auc, f"[{lo:.3f}, {hi:.3f}]"
        out[f"{cohort}_sig"] = "YES" if lo > 0.5 else "no"
    return out, lr

rows, models = [], {}
for cols, label in [(list(ps_t.columns), "PRIMARY: all 15 pathways"),
                    (BCR_LITERATURE, "SECONDARY: BCR-literature 5"),
                    (['Proliferation', 'Cell_Cycle'], "SENSITIVITY: prolif+cellcycle")]:
    res, mdl = eval_config(cols, label)
    rows.append(res); models[label] = (cols, mdl)
    print(res)

res_df = pd.DataFrame(rows)
display(res_df)

{'config': 'PRIMARY: all 15 pathways', 'C': 0.1, 'cv_auc': np.float64(0.5530069052102949), 'test_auc': 0.6813063063063064, 'GSE70769_auc': 0.5659863945578232, 'GSE70769_ci': '[0.446, 0.685]', 'GSE70769_sig': 'no', 'GSE54460_auc': 0.6813725490196079, 'GSE54460_ci': '[0.495, 0.844]', 'GSE54460_sig': 'no'}
{'config': 'SECONDARY: BCR-literature 5', 'C': 0.1, 'cv_auc': np.float64(0.5417263025737602), 'test_auc': 0.6126126126126126, 'GSE70769_auc': 0.7129251700680272, 'GSE70769_ci': '[0.606, 0.812]', 'GSE70769_sig': 'YES', 'GSE54460_auc': 0.553921568627451, 'GSE54460_ci': '[0.335, 0.765]', 'GSE54460_sig': 'no'}
{'config': 'SENSITIVITY: prolif+cellcycle', 'C': 1, 'cv_auc': np.float64(0.541908349026993), 'test_auc': 0.6148648648648648, 'GSE70769_auc': 0.6975056689342403, 'GSE70769_ci': '[0.585, 0.804]', 'GSE70769_sig': 'YES', 'GSE54460_auc': 0.6053921568627452, 'GSE54460_ci': '[0.386, 0.809]', 'GSE54460_sig': 'no'}


,config,C,cv_auc,test_auc,GSE70769_auc,GSE70769_ci,GSE70769_sig,GSE54460_auc,GSE54460_ci,GSE54460_sig
0,PRIMARY: all 15 pathways,0.1,0.553007,0.681306,0.565986,"[0.446, 0.685]",no,0.681373,"[0.495, 0.844]",no
1,SECONDARY: BCR-literature 5,0.1,0.541726,0.612613,0.712925,"[0.606, 0.812]",YES,0.553922,"[0.335, 0.765]",no
2,SENSITIVITY: prolif+cellcycle,1.0,0.541908,0.614865,0.697506,"[0.585, 0.804]",YES,0.605392,"[0.386, 0.809]",no


In [8]:
# --- Operating points on GSE70769 for the BCR-literature model ---
# Probability scales differ across platforms, so a train-derived threshold does not
# transfer. Standard biomarker-validation practice: report the threshold-free AUC as
# primary, plus descriptive operating points read off the EXTERNAL ROC curve.
from sklearn.metrics import roc_curve

cols, mdl = models["SECONDARY: BCR-literature 5"]
p_ext = mdl.predict_proba(ps_e[cols])[:, 1]
yv = y_ext.values
fpr, tpr, thr = roc_curve(yv, p_ext)

op_rows = []
for spec_target in (0.80, 0.90):
    i = np.argmin(np.abs((1 - fpr) - spec_target))
    op_rows.append({"operating_point": f"Sens@Spec={spec_target:.0%}",
                    "threshold": thr[i], "sensitivity": tpr[i],
                    "specificity": 1 - fpr[i]})

j = np.argmax(tpr - fpr)
pred = (p_ext >= thr[j]).astype(int)
tp = int(((pred == 1) & (yv == 1)).sum()); fn = int(((pred == 0) & (yv == 1)).sum())
tn = int(((pred == 0) & (yv == 0)).sum()); fp = int(((pred == 1) & (yv == 0)).sum())
op_rows.append({"operating_point": "Youden (descriptive)", "threshold": thr[j],
                "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp)})
op = pd.DataFrame(op_rows)
display(op)
print(f"Youden confusion: TP={tp} FN={fn} TN={tn} FP={fp}")
print(f"PPV={tp / max(tp + fp, 1):.3f}  NPV={tn / max(tn + fn, 1):.3f}")

# Coefficients
coefs = pd.Series(mdl.coef_[0], index=cols).sort_values(key=abs, ascending=False)
display(coefs.to_frame("coefficient"))

res_df.to_csv(TABLES / "pathway_external_final.csv", index=False)
coefs.to_csv(TABLES / "pathway_model_coefficients.csv")
op.to_csv(TABLES / "pathway_operating_points.csv", index=False)
print("Saved -> pathway_external_final.csv, pathway_model_coefficients.csv, pathway_operating_points.csv")


,operating_point,threshold,sensitivity,specificity
0,Sens@Spec=80%,0.351342,0.377778,0.775510
1,Sens@Spec=90%,0.356494,0.244444,0.897959
2,Youden (descriptive),0.329091,0.911111,0.448980


Youden confusion: TP=41 FN=4 TN=22 FP=27
PPV=0.603  NPV=0.846


,coefficient
Cell_Cycle,-0.382891
DNA_Repair,0.307311
Proliferation,0.230723
Decipher,-0.228147
Prolaris,0.076679


Saved -> pathway_external_final.csv, pathway_model_coefficients.csv, pathway_operating_points.csv
